# LLM Quantization Workshop: Qwen2.5-0.5B

In this workshop, we'll:
1. Quantize Qwen2.5-0.5B using llm-compressor
2. Compare original vs quantized models
3. Deploy with vllm and benchmark performance

**Model:** [Qwen2.5-0.5B](https://huggingface.co/Qwen/Qwen2.5-0.5B) - 500M parameter model

## Setup: Install Dependencies

If not already installed in your environment:

In [ ]:
# Uncomment if needed
# !pip install llm-compressor vllm transformers datasets torch

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from llmcompressor.transformers import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier
import os

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

## Part 1: Understanding the Baseline Model

First, let's load the original Qwen2.5-0.5B model and see its characteristics.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"

print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

print(f"\nModel loaded!")
print(f"Parameters: {model.num_parameters():,}")
print(f"Dtype: {model.dtype}")

### Check Model Size on Disk

In [ ]:
import os

def get_model_size(model_path):
    """Get total size of model files in GB"""
    total_size = 0
    for root, dirs, files in os.walk(model_path):
        for f in files:
            fp = os.path.join(root, f)
            if os.path.exists(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024**3)  # Convert to GB

# Model will be cached in huggingface cache
cache_dir = os.path.expanduser("~/.cache/huggingface/hub")
print(f"Models cached in: {cache_dir}")

### Test Inference with Original Model

In [ ]:
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print(f"Generating with original model...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nGenerated: {generated_text[len(prompt):]}")

## Part 2: Quantization with llm-compressor

Now let's quantize the model using GPTQ (Generative Pre-trained Quantization).

### What is GPTQ?
- Reduces model weights to 4-bit or 8-bit integers
- Uses calibration data to minimize accuracy loss
- Dramatically reduces memory footprint
- Enables faster inference

In [ ]:
# Clean up to free GPU memory
del model
torch.cuda.empty_cache()

### Configure Quantization Recipe

In [ ]:
# GPTQ quantization recipe for INT4 (W4A16)
recipe = """
quant_stage:
    quant_modifiers:
        GPTQModifier:
            sequential_update: false
            ignore: ["lm_head"]
            config_groups:
                group_0:
                    weights:
                        num_bits: 4
                        type: "int"
                        symmetric: true
                        strategy: "channel"
                    targets: ["Linear"]
"""

print("Quantization recipe configured: INT4 (4-bit weights, 16-bit activations)")

### Prepare Calibration Data

GPTQ needs sample data to calibrate the quantization. We'll use a small subset from the C4 dataset.

In [ ]:
from datasets import load_dataset

# Load calibration dataset
print("Loading calibration dataset...")
calibration_dataset = load_dataset(
    "allenai/c4",
    data_files="en/c4-train.00000-of-01024.json.gz",
    split="train"
).select(range(256))  # Use 256 samples for calibration

print(f"Calibration samples: {len(calibration_dataset)}")
print(f"Sample text: {calibration_dataset[0]['text'][:200]}...")

### Run One-Shot Quantization

This will take ~2-5 minutes on an L40S.

In [ ]:
import time

output_dir = "./qwen2.5-0.5b-gptq-int4"

print("Starting quantization...\n")
start_time = time.time()

oneshot(
    model=model_id,
    dataset=calibration_dataset,
    recipe=recipe,
    output_dir=output_dir,
    max_seq_length=2048,
    num_calibration_samples=256,
)

elapsed = time.time() - start_time
print(f"\n✓ Quantization complete in {elapsed:.1f} seconds!")
print(f"Quantized model saved to: {output_dir}")

### Compare Model Sizes

In [ ]:
quantized_size = get_model_size(output_dir)

# Estimate original size (FP16: 2 bytes per param)
num_params = 494033920  # Qwen2.5-0.5B params
original_size_gb = (num_params * 2) / (1024**3)

print(f"Original model (FP16): ~{original_size_gb:.2f} GB")
print(f"Quantized model (INT4): {quantized_size:.2f} GB")
print(f"\nCompression ratio: {original_size_gb/quantized_size:.2f}x")
print(f"Size reduction: {(1 - quantized_size/original_size_gb)*100:.1f}%")

## Part 3: Testing the Quantized Model

Let's load the quantized model and compare inference quality.

In [ ]:
print("Loading quantized model...")
quantized_model = AutoModelForCausalLM.from_pretrained(
    output_dir,
    device_map="auto",
    torch_dtype="auto"
)

print(f"\nQuantized model loaded!")
print(f"Parameters: {quantized_model.num_parameters():,}")

In [ ]:
# Test with same prompt
prompt = "The future of artificial intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(quantized_model.device)

print(f"Generating with quantized model...\n")
outputs = quantized_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {prompt}")
print(f"\nGenerated: {generated_text[len(prompt):]}")

## Part 4: Deploy with vLLM

Now let's deploy both models with vLLM and benchmark performance.

In [ ]:
from vllm import LLM, SamplingParams

# Clean up transformers models
del quantized_model
torch.cuda.empty_cache()

### Load Original Model in vLLM

In [ ]:
print("Loading original model in vLLM...")
llm_original = LLM(
    model=model_id,
    dtype="float16",
    gpu_memory_utilization=0.4
)
print("✓ Original model loaded")

### Load Quantized Model in vLLM

In [ ]:
print("Loading quantized model in vLLM...")
llm_quantized = LLM(
    model=output_dir,
    quantization="gptq",
    gpu_memory_utilization=0.4
)
print("✓ Quantized model loaded")

### Benchmark Throughput

In [ ]:
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=100
)

# Test prompts
prompts = [
    "The future of artificial intelligence is",
    "Machine learning models are",
    "Quantum computing will",
    "The key to sustainable energy is",
    "The most important skill for a developer is"
] * 10  # 50 total prompts

print(f"Running benchmark with {len(prompts)} prompts...\n")

In [ ]:
# Benchmark original model
print("Benchmarking original model...")
start = time.time()
outputs_original = llm_original.generate(prompts, sampling_params)
time_original = time.time() - start

throughput_original = len(prompts) / time_original
print(f"Original: {time_original:.2f}s, {throughput_original:.2f} req/s")

In [ ]:
# Benchmark quantized model
print("Benchmarking quantized model...")
start = time.time()
outputs_quantized = llm_quantized.generate(prompts, sampling_params)
time_quantized = time.time() - start

throughput_quantized = len(prompts) / time_quantized
print(f"Quantized: {time_quantized:.2f}s, {throughput_quantized:.2f} req/s")

### Performance Summary

In [ ]:
print("="*60)
print("PERFORMANCE COMPARISON")
print("="*60)
print(f"\nModel Size:")
print(f"  Original (FP16):  ~{original_size_gb:.2f} GB")
print(f"  Quantized (INT4): {quantized_size:.2f} GB")
print(f"  Reduction:        {(1 - quantized_size/original_size_gb)*100:.1f}%")

print(f"\nThroughput:")
print(f"  Original:         {throughput_original:.2f} req/s")
print(f"  Quantized:        {throughput_quantized:.2f} req/s")
print(f"  Speedup:          {throughput_quantized/throughput_original:.2f}x")

print(f"\nLatency per request:")
print(f"  Original:         {1000/throughput_original:.1f} ms")
print(f"  Quantized:        {1000/throughput_quantized:.1f} ms")
print("="*60)

## Part 5: Experimentation

Try modifying the quantization configuration:

### Try INT8 instead of INT4

Change `num_bits: 4` to `num_bits: 8` in the recipe and re-run quantization.

### Try Different Calibration Data

Use a different dataset or more/fewer samples.

### Questions to Explore:
- How does INT8 vs INT4 affect model size?
- What's the quality vs speed tradeoff?
- How much calibration data do we actually need?

## Bonus: Check GPU Memory Usage

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

## Summary

In this workshop, you:
1. ✓ Quantized Qwen2.5-0.5B from FP16 to INT4 using llm-compressor
2. ✓ Reduced model size by ~75%
3. ✓ Deployed with vLLM and measured performance gains
4. ✓ Understood the quantization/quality tradeoff

## Next Steps

- Try larger models (Qwen2.5-1.5B, Llama 3.2-3B)
- Experiment with different quantization techniques (AWQ, SmoothQuant)
- Deploy in production with vLLM OpenAI-compatible server
- Measure quality impact with perplexity benchmarks

## Resources

- [llm-compressor docs](https://github.com/vllm-project/llm-compressor)
- [vLLM docs](https://docs.vllm.ai/)
- [GPTQ paper](https://arxiv.org/abs/2210.17323)